# Day 17 — Django REST API Setup and MySQL Integration

**Project:** IPL Player Performance Analytics and Prediction  
**Goal:** Prepare the Django API foundation, verify MySQL connectivity through Django settings, install/configure Django REST Framework, and create automatic documentation based on actual checks.

## Day 17 scope
- Inspect the current project structure and relevant Day 15/16 outputs.
- Confirm required Python packages are available.
- Check `.env` loading without printing secrets.
- Prepare a Django project skeleton only if it does not already exist.
- Configure Django for MySQL via environment settings.
- Run Django system checks and a read-only database check.
- Generate `docs/day17_django_api_setup.md` and `reports/day17/task_status.csv` automatically.

**Important:** This notebook does not create the prediction/EDA endpoints yet, does not migrate models, and does not deploy to AWS. It avoids overwriting existing Django source files; where files already exist, it reports their status instead of replacing them.

## 1. Detect the project root and inspect expected folders
Run from the project directory or a notebook inside it. The project root is detected by locating `data/processed`.

In [1]:
from pathlib import Path
import os, sys, json, subprocess, importlib.util, traceback
from datetime import datetime

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "processed").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate project root (expected data/processed). "
        "Run this notebook from within the IPL project directory."
    )

PROJECT_ROOT = find_project_root()
DJANGO_ROOT = PROJECT_ROOT / "django_api"
CONFIG_DIR = DJANGO_ROOT / "config"
API_DIR = DJANGO_ROOT / "api"
DOCS_DIR = PROJECT_ROOT / "docs"
REPORT_DIR = PROJECT_ROOT / "reports" / "day17"
for folder in (DOCS_DIR, REPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

RUN_STARTED_AT = datetime.now().astimezone().isoformat(timespec="seconds")
print("Project root:", PROJECT_ROOT)
print("Python executable:", sys.executable)
print("Django root exists:", DJANGO_ROOT.exists())
print("Run started:", RUN_STARTED_AT)

Project root: D:\Git\Personal\ipl-player-performance
Python executable: c:\Users\Vinod\anaconda3\envs\ipl-player-performance\python.exe
Django root exists: True
Run started: 2026-10-21T23:10:11+05:30


## 2. Inspect previous-day outputs and project structure
This checks existence only; it does not assume earlier notebooks ran successfully.

In [2]:
expected_paths = [
    PROJECT_ROOT / "docs" / "day15_mysql_setup_schema_preparation.md",
    PROJECT_ROOT / "docs" / "day16_mysql_data_loading_validation.md",
    PROJECT_ROOT / "reports" / "day16" / "database_validation.csv",
    PROJECT_ROOT / "reports" / "day16" / "task_status.csv",
    PROJECT_ROOT / "data" / "processed" / "batting_player_match.csv",
    PROJECT_ROOT / "data" / "processed" / "bowling_player_match.csv",
    PROJECT_ROOT / "data" / "processed" / "match_metadata.csv",
]
previous_outputs = []
for path in expected_paths:
    previous_outputs.append({
        "path": str(path.relative_to(PROJECT_ROOT)),
        "exists": path.exists(),
        "type": "directory" if path.is_dir() else ("file" if path.is_file() else "missing")
    })

import pandas as pd
previous_outputs_df = pd.DataFrame(previous_outputs)
previous_outputs_df.to_csv(REPORT_DIR / "previous_output_check.csv", index=False)
display(previous_outputs_df)

for relative in ["django_api", "django_api/config", "django_api/api", "tests", "deployment"]:
    p = PROJECT_ROOT / relative
    print(f"{relative}: {'EXISTS' if p.exists() else 'MISSING'}")

,path,exists,type
0,docs\day15_mysql_setup_schema_preparation.md,True,file
1,docs\day16_mysql_data_loading_validation.md,True,file
2,reports\day16\database_validation.csv,True,file
3,reports\day16\task_status.csv,True,file
4,data\processed\batting_player_match.csv,True,file
5,data\processed\bowling_player_match.csv,True,file
6,data\processed\match_metadata.csv,True,file


django_api: EXISTS
django_api/config: EXISTS
django_api/api: EXISTS
tests: EXISTS
deployment: EXISTS


## 3. Check package availability
If packages are missing, install them from the active `ipl-player-performance` environment using the terminal, then rerun this cell.

In [3]:
required_packages = ["django", "rest_framework", "sqlalchemy", "pymysql", "dotenv"]
package_status = []
for package in required_packages:
    available = importlib.util.find_spec(package) is not None
    package_status.append({"package": package, "available": available})

package_df = pd.DataFrame(package_status)
package_df.to_csv(REPORT_DIR / "package_availability.csv", index=False)
display(package_df)

missing_packages = package_df.loc[~package_df["available"], "package"].tolist()
if missing_packages:
    print("Packages missing from this kernel:", ", ".join(missing_packages))
    print("Install Django/API packages in the active environment, for example:")
    print("python -m pip install django djangorestframework django-environ pymysql")
else:
    print("All checked packages are importable.")

,package,available
0,django,True
1,rest_framework,True
2,sqlalchemy,True
3,pymysql,True
4,dotenv,True


All checked packages are importable.


## 4. Load `.env` and safely inspect settings
This reports whether the database URL is MySQL without showing credentials.

In [4]:
from dotenv import load_dotenv
from sqlalchemy.engine import make_url

env_path = PROJECT_ROOT / ".env"
if env_path.exists():
    os.environ.pop("DATABASE_URL", None)
    load_dotenv(env_path, override=False)
else:
    print("WARNING: Project .env file not found.")

database_url = os.getenv("DATABASE_URL")
db_config_status = "MISSING"
db_driver = db_host = db_port = db_name = None

if database_url:
    parsed = make_url(database_url)
    db_driver = parsed.drivername
    db_host = parsed.host
    db_port = parsed.port
    db_name = parsed.database
    db_config_status = "PASS" if db_driver.startswith("mysql") else "REVIEW"

print("DATABASE_URL present:", bool(database_url))
print("Database driver:", db_driver)
print("Database host:", db_host)
print("Database port:", db_port)
print("Database name:", db_name)
print("Database URL status:", db_config_status)

if db_config_status != "PASS":
    print("Update .env to use a MySQL SQLAlchemy URL before continuing.")

DATABASE_URL present: True
Database driver: mysql+pymysql
Database host: localhost
Database port: 3306
Database name: ipl_analytics
Database URL status: PASS


## 5. Create the actual Django project safely

This step creates `django_api/manage.py` and the `django_api/config/` project package if they are missing. It uses a temporary folder to generate a clean Django skeleton, then copies only files that do not already exist. Existing files are never overwritten.

If `django_api` contains an incompatible partial setup, the notebook records that and reports the exact conflict rather than silently pretending Django was initialized.

In [5]:
import tempfile
import shutil

manage_py = DJANGO_ROOT / "manage.py"
settings_py = CONFIG_DIR / "settings.py"
urls_py = CONFIG_DIR / "urls.py"
wsgi_py = CONFIG_DIR / "wsgi.py"
asgi_py = CONFIG_DIR / "asgi.py"

def copy_missing_tree(source_dir, destination_dir):
    destination_dir.mkdir(parents=True, exist_ok=True)
    copied, skipped = [], []
    for source in source_dir.rglob("*"):
        if source.is_dir():
            continue
        relative = source.relative_to(source_dir)
        destination = destination_dir / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.exists():
            skipped.append(str(destination.relative_to(PROJECT_ROOT)))
        else:
            shutil.copy2(source, destination)
            copied.append(str(destination.relative_to(PROJECT_ROOT)))
    return copied, skipped

django_available = importlib.util.find_spec("django") is not None
skeleton_action = "NOT_RUN"
skeleton_copied = []
skeleton_skipped = []

if not django_available:
    skeleton_action = "BLOCKED_DJANGO_NOT_INSTALLED"
    print("Django is not installed in this notebook's Python environment.")
    print("Install it in the active environment: python -m pip install django djangorestframework django-environ pymysql")
else:
    DJANGO_ROOT.mkdir(parents=True, exist_ok=True)
    if manage_py.exists() and settings_py.exists():
        skeleton_action = "EXISTING_DJANGO_PROJECT_PRESERVED"
        print("Django project already exists; preserving existing files.")
    else:
        with tempfile.TemporaryDirectory(prefix="ipl_django_project_") as temp:
            temp_root = Path(temp)
            generated_root = temp_root / "generated"
            result = subprocess.run(
                [sys.executable, "-m", "django", "startproject", "config", str(generated_root)],
                cwd=str(temp_root), capture_output=True, text=True
            )
            print("Django startproject return code:", result.returncode)
            if result.stdout.strip():
                print(result.stdout.strip())
            if result.stderr.strip():
                print(result.stderr.strip())
            if result.returncode == 0:
                skeleton_copied, skeleton_skipped = copy_missing_tree(generated_root, DJANGO_ROOT)
                if manage_py.exists() and settings_py.exists():
                    skeleton_action = "CREATED_MISSING_PROJECT_FILES"
                else:
                    skeleton_action = "PARTIAL_CONFLICT_REVIEW"
            else:
                skeleton_action = "FAILED"

print("Django project action:", skeleton_action)
print("Files created:", skeleton_copied)
print("Existing files preserved:", skeleton_skipped)
print("manage.py exists:", manage_py.exists())
print("settings.py exists:", settings_py.exists())

Django project already exists; preserving existing files.
Django project action: EXISTING_DJANGO_PROJECT_PRESERVED
Files created: []
Existing files preserved: []
manage.py exists: True
settings.py exists: True


## 6. Create the actual `api` Django app safely

If the project is initialized and the app's `apps.py` is missing, this generates a Django app in a temporary folder and copies only missing files into `django_api/api/`. It does not overwrite existing files.

In [6]:
api_app_action = "NOT_RUN"
api_copied, api_skipped = [], []

if not django_available:
    api_app_action = "BLOCKED_DJANGO_NOT_INSTALLED"
elif not manage_py.exists() or not settings_py.exists():
    api_app_action = "BLOCKED_DJANGO_PROJECT_NOT_READY"
elif (API_DIR / "apps.py").exists():
    api_app_action = "EXISTING_API_APP_PRESERVED"
else:
    API_DIR.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="ipl_django_app_") as temp:
        temp_root = Path(temp)
        generated_app = temp_root / "api"
        result = subprocess.run(
            [sys.executable, str(manage_py), "startapp", "api", str(generated_app)],
            cwd=str(DJANGO_ROOT), capture_output=True, text=True
        )
        print("Django startapp return code:", result.returncode)
        if result.stdout.strip():
            print(result.stdout.strip())
        if result.stderr.strip():
            print(result.stderr.strip())
        if result.returncode == 0 and generated_app.exists():
            api_copied, api_skipped = copy_missing_tree(generated_app, API_DIR)
            api_app_action = "CREATED_MISSING_API_APP_FILES" if (API_DIR / "apps.py").exists() else "PARTIAL_CONFLICT_REVIEW"
        else:
            api_app_action = "FAILED"

print("API app action:", api_app_action)
print("API app files created:", api_copied)
print("Existing API files preserved:", api_skipped)
print("api/apps.py exists:", (API_DIR / "apps.py").exists())

Django startapp return code: 1
CommandError: 'api' conflicts with the name of an existing Python module and cannot be used as an app name. Please try another name.
API app action: FAILED
API app files created: []
Existing API files preserved: []
api/apps.py exists: False


## 7. Configure Django settings for MySQL (manual-safe patch)
To avoid overwriting a user's existing settings, this cell writes a configuration snippet to `django_api/config/day17_settings_patch.txt`. Review and merge the snippet into `settings.py` manually. It does not change settings.py automatically.

In [7]:
settings_patch = '''
# Day 17 MySQL + Django REST Framework settings snippet.
# Merge into django_api/config/settings.py after reviewing existing settings.

import os
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy.engine import make_url

PROJECT_ROOT = Path(__file__).resolve().parents[2]
load_dotenv(PROJECT_ROOT / ".env", override=False)

INSTALLED_APPS += [
    "rest_framework",
    "api",
]

# If the project uses a different database URL parser, adapt this section carefully.
DATABASE_URL = os.environ.get("DATABASE_URL")
if not DATABASE_URL:
    raise RuntimeError("DATABASE_URL is not configured")

db_url = make_url(DATABASE_URL)
if not db_url.drivername.startswith("mysql"):
    raise RuntimeError("Expected a MySQL DATABASE_URL")

DATABASES = {
    "default": {
        "ENGINE": "django.db.backends.mysql",
        "NAME": db_url.database,
        "USER": db_url.username,
        "PASSWORD": db_url.password,
        "HOST": db_url.host or "localhost",
        "PORT": str(db_url.port or 3306),
        "OPTIONS": {"charset": "utf8mb4"},
        "CONN_MAX_AGE": 60,
    }
}

ALLOWED_HOSTS = [
    host.strip()
    for host in os.getenv("ALLOWED_HOSTS", "127.0.0.1,localhost").split(",")
    if host.strip()
]
DEBUG = os.getenv("DEBUG", "False").lower() in {"1", "true", "yes"}
'''
patch_path = CONFIG_DIR / "day17_settings_patch.txt"
if CONFIG_DIR.exists():
    if not patch_path.exists():
        patch_path.write_text(settings_patch.strip() + "\n", encoding="utf-8")
        patch_status = "CREATED_REVIEW_BEFORE_MERGING"
    else:
        patch_status = "EXISTING_PATCH_PRESERVED"
else:
    patch_status = "SKIPPED_CONFIG_DIR_MISSING"

print("Settings patch status:", patch_status)
if patch_path.exists():
    print("Patch file:", patch_path)

Settings patch status: EXISTING_PATCH_PRESERVED
Patch file: D:\Git\Personal\ipl-player-performance\django_api\config\day17_settings_patch.txt


## 8. Django system check and read-only database check
This step runs only if a complete Django project exists. The system check may fail until the settings patch is reviewed and merged. No migrations are run on Day 17.

In [8]:
django_check_status = "SKIPPED"
django_check_output = ""
db_check_status = "SKIPPED"
db_check_output = ""

if django_available and manage_py.exists() and settings_py.exists():
    result = subprocess.run(
        [sys.executable, str(manage_py), "check"],
        cwd=str(DJANGO_ROOT),
        capture_output=True, text=True
    )
    django_check_status = "PASS" if result.returncode == 0 else "FAIL"
    django_check_output = (result.stdout + "\n" + result.stderr).strip()
    print("Django check:", django_check_status)
    print(django_check_output[-3000:] if django_check_output else "(no output)")

    # Test DB URL directly using SQLAlchemy without changing database contents.
    if db_config_status == "PASS":
        try:
            from sqlalchemy import create_engine, text
            engine = create_engine(
                database_url,
                pool_pre_ping=True,
                connect_args={"connect_timeout": 5}
            )
            with engine.connect() as conn:
                db_ping = conn.execute(text("SELECT 1")).scalar_one()
                db_version = conn.execute(text("SELECT VERSION()")).scalar_one()
            db_check_status = "PASS" if db_ping == 1 else "FAIL"
            db_check_output = f"SELECT 1={db_ping}; MySQL version={db_version}"
        except Exception as exc:
            db_check_status = "FAIL"
            db_check_output = f"{type(exc).__name__}: {exc}"
else:
    print("Skipped Django check because Django project files are not ready.")

print("Read-only DB check:", db_check_status)
print(db_check_output)

Django check: PASS
System check identified no issues (0 silenced).
Read-only DB check: PASS
SELECT 1=1; MySQL version=8.0.46


## 9. Generate Day 17 documentation automatically
The generated Markdown file records the real statuses above. A setup-only day is not marked complete if Django checks or the database check failed.

In [9]:
task_rows = [
    {"task": "Previous day outputs inspection", "status": "PASS" if previous_outputs_df["exists"].all() else "REVIEW"},
    {"task": "Required package availability", "status": "PASS" if not missing_packages else "REVIEW"},
    {"task": "MySQL URL configuration", "status": db_config_status},
    {"task": "Django project created/present", "status": "PASS" if skeleton_action in {"CREATED_MISSING_PROJECT_FILES", "EXISTING_DJANGO_PROJECT_PRESERVED"} else "REVIEW"},
    {"task": "Django api app created/present", "status": "PASS" if api_app_action in {"CREATED_MISSING_API_APP_FILES", "EXISTING_API_APP_PRESERVED"} else "REVIEW"},
    {"task": "Settings patch", "status": "PASS" if patch_status in {"CREATED_REVIEW_BEFORE_MERGING", "EXISTING_PATCH_PRESERVED"} else "REVIEW"},
    {"task": "Django system check", "status": django_check_status},
    {"task": "Read-only MySQL connection check", "status": db_check_status},
]
task_status = pd.DataFrame(task_rows)
all_pass = task_status["status"].eq("PASS").all()
overall_status = "PASS" if all_pass else "REVIEW REQUIRED"
task_status.loc[len(task_status)] = {"task": "OVERALL", "status": overall_status}
task_status.to_csv(REPORT_DIR / "task_status.csv", index=False)

def md_table(df):
    if df.empty:
        return "_No data._"
    cols = [str(c) for c in df.columns]
    vals = df.fillna("").astype(str).values.tolist()
    widths = [max([len(cols[i])] + [len(row[i]) for row in vals]) for i in range(len(cols))]
    lines = [
        "| " + " | ".join(cols[i].ljust(widths[i]) for i in range(len(cols))) + " |",
        "| " + " | ".join("-" * widths[i] for i in range(len(cols))) + " |",
    ]
    lines += ["| " + " | ".join(row[i].ljust(widths[i]) for i in range(len(cols))) + " |" for row in vals]
    return "\n".join(lines)

doc = [
    "# Day 17 — Django REST API Setup and MySQL Integration",
    "",
    f"- Run started: `{RUN_STARTED_AT}`",
    f"- Project root: `{PROJECT_ROOT}`",
    f"- Overall status: **{overall_status}**",
    "",
    "## Goal",
    "Prepare the Django REST API foundation and verify environment/database configuration before building endpoints.",
    "",
    "## Task status",
    md_table(task_status),
    "",
    "## Previous output inspection",
    md_table(previous_outputs_df),
    "",
    "## Package availability",
    md_table(package_df),
    "",
    "## Django setup actions",
    f"- Django project action: `{skeleton_action}`",
    f"- Django project files created: `{skeleton_copied}`",
    f"- API app action: `{api_app_action}`",
    f"- API app files created: `{api_copied}`",
    f"- Settings patch: `{patch_status}`",
    f"- Settings patch file: `{patch_path.relative_to(PROJECT_ROOT) if patch_path.exists() else 'not created'}`",
    "",
    "## Checks",
    f"- Django system check: `{django_check_status}`",
    f"- Django output: `{django_check_output[:1500] if django_check_output else 'No output / not run'}`",
    f"- Read-only MySQL check: `{db_check_status}`",
    f"- MySQL output: `{db_check_output}`",
    "",
    "## Important limitations",
    "- No database migrations were run.",
    "- No EDA, prediction, batch, or ingestion API endpoints were created.",
    "- Existing Django source files were not overwritten.",
    "- Review and merge the generated settings patch manually before running Django.",
    "- Do not commit `.env` or credentials.",
    "",
    "## Next day",
    "Day 18: implement and test player list/detail endpoints against the verified database schema.",
    "",
]
doc_path = DOCS_DIR / "day17_django_api_setup.md"
doc_path.write_text("\n".join(doc), encoding="utf-8")
print("Generated documentation:", doc_path)
print("Generated task status:", REPORT_DIR / "task_status.csv")
print("Overall status:", overall_status)
print(doc_path.read_text(encoding="utf-8"))

Generated documentation: D:\Git\Personal\ipl-player-performance\docs\day17_django_api_setup.md
Generated task status: D:\Git\Personal\ipl-player-performance\reports\day17\task_status.csv
Overall status: REVIEW REQUIRED
# Day 17 — Django REST API Setup and MySQL Integration

- Run started: `2026-10-21T23:10:11+05:30`
- Project root: `D:\Git\Personal\ipl-player-performance`
- Overall status: **REVIEW REQUIRED**

## Goal
Prepare the Django REST API foundation and verify environment/database configuration before building endpoints.

## Task status
| task                             | status          |
| -------------------------------- | --------------- |
| Previous day outputs inspection  | PASS            |
| Required package availability    | PASS            |
| MySQL URL configuration          | PASS            |
| Django project created/present   | PASS            |
| Django api app created/present   | REVIEW          |
| Settings patch                   | PASS            |
| Django s

## 10. Day-end checklist
- [ ] Day 15 and Day 16 reports exist and were reviewed.
- [ ] MySQL connection check returns `SELECT 1 = 1`.
- [ ] Django and Django REST Framework are installed in this kernel's environment.
- [ ] Existing Django files were preserved.
- [ ] Review `django_api/config/day17_settings_patch.txt` and merge into settings manually if applicable.
- [ ] Django `manage.py check` passes after settings are correctly configured.
- [ ] `docs/day17_django_api_setup.md` and `reports/day17/task_status.csv` were generated.
- [ ] `.env` is ignored by Git and no credentials were printed or committed.

**Stop after reviewing the generated report.** Do not start Day 18 until the MySQL and Django configuration checks are passing.